# Incremental Embedding Updates

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qdrant/examples/blob/master/temporal-data-drift/sync_raw_data_to_embeddings.ipynb)

Qdrant documentation [lives on GitHub](https://github.com/qdrant/landing_page), consisting mainly of markdown pages with embedded code snippets and visuals.  
As any other documentation of an evolving product, it's not static: for example, releases add new features that have to be documented.

So, raw data in markdowns changes with time, and users searching across our documentation expect to find the latest state of it.  
If search over documentation uses vectors, and our certainly does, it requires additional setup and maintenance to fulfill this expectation.

## Vectors <-> Raw Data

Vectors are a transformation of raw data.
This transformation is not happening by itself when raw data changes. Unless vectors are updated proactively, documentation search would run against embeddings of text that no longer exists, a drift growing with every edit.
We have to set up a re-embedding process, syncing vectors with raw data changes.

The simplest option would be re-embedding the whole dataset on a schedule. However, it's expensive, especially as the corpus grows, and for the majority of the data points the inference cost will be paid for no reason. 

This tutorial provides a simple pipeline that, set from the beginning, detects changes in your text data and executes incremental embedding updates.  
It reconciles a complete, current list of Qdrant documentation chunks with a Qdrant collection. Each run:

1. leaves unchanged chunks untouched,
2. re-embeds changed text,
3. reuses a vector when text changes location,
4. adds new text, and
5. deletes text absent from the authoritative source list.

The pattern applies when your chunking is deterministic and enumerating the current source is inexpensive.

## Prerequisites
This notebook uses Qdrant Cloud and Qdrant Cloud Free Tier Inference.  
Create a Free Tier Qdrant Cloud cluster and set QDRANT_URL and QDRANT_API_KEY in the notebook environment.  

In [13]:
%pip install -q "qdrant-client>=1.18"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from qdrant_client import QdrantClient, models

QDRANT_URL = os.getenv("QDRANT_URL", "https://YOUR-CLUSTER.cloud.qdrant.io:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "YOUR-API-KEY")

if not QDRANT_URL or not QDRANT_API_KEY:
    raise RuntimeError(
        "Set QDRANT_URL and QDRANT_API_KEY for a Qdrant Cloud cluster before running this notebook."
    )

client = QdrantClient(
    url=QDRANT_URL, 
    api_key=QDRANT_API_KEY, 
    cloud_inference=True
)

## The Data: Qdrant Documentation

Let's look at the [operations tutorials](https://qdrant.tech/documentation/tutorials-operations/) tab on qdrant.tech. Here's an example of a real change: this tutorial will become a part of this tab, so our collection of vectors used for documentation search will have to be updated.

Let's consider a simple documentation hierarchy:

- We have one page behind one `url`: https://qdrant.tech/documentation/tutorials-operations/secure-qdrant
- A page consists of sections. A **section** is everything between one heading and the next. For example, the ["Step 2: Enable TLS" section](https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/#step-2-enable-tls)
    - A section is marked by an `anchor`, generated from the heading text: "Step 2: Enable TLS" -> the `#step-2-enable-tls` part of the link "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/#step-2-enable-tls".

For vector search, we break down documentation using this hierarchy. One vector = one section chunk.

### Chunking Sections

Some sections might not fit the embedding model context window limit (how big of a text it can represent). Simplest approach: split a section in under-context-window-sized pieces, numbered `0, 1, 2…`. For minimal hierarchy awareness, a chunk keeps its section heading prepended.

So one page produces a set of chunks of the form: `{url, anchor, chunk_num, text}`.

In [15]:
CHUNKS = [  # three tutorials: secure-qdrant, migration, time-based-sharding
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
  "anchor": "prerequisites",
  "chunk_num": 0,
  "text": "Prerequisites - Docker and Docker Compose installed - `curl` available in your terminal - mkcert for generating a local self-signed certificate (installation instructions) - TLS requires Qdrant 1.2 or later, API key authentication requires Qdrant 1.2 or later, and granular access API keys (JWT) require Qdrant 1.9 or later. This tutorial uses the latest Qdrant image, which includes all these features. ---"
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
  "anchor": "secure-a-self-hosted-qdrant-instance",
  "chunk_num": 0,
  "text": "Secure a Self-Hosted Qdrant Instance | Time: 45 min | Level: Intermediate | | --- | ----------- | Qdrant offers a comprehensive set of security and access control features that enable you to protect your data and control access at multiple levels. By default, these features are enabled on Qdrant Cloud deployments. However, self-hosted Qdrant deployments default to no authentication and no encryption: every interface on the host is reachable without a key or password. For self-hosted instances, it is crucial to secure your instance before connecting it to any network. This tutorial walks through securing a self-hosted Qdrant instance step by step. You will: - **Enable TLS** to encrypt traffic between clients and your Qdrant instance. - **Set up an admin API key** to require authentication for all requests. - **Restrict consumers with a read-only key** to prevent unintended writes. - **Issue granular access API keys** to scope permissions to specific collections."
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
  "anchor": "secure-a-self-hosted-qdrant-instance",
  "chunk_num": 1,
  "text": "Secure a Self-Hosted Qdrant Instance > Qdrant Cloud deployments are always secure by default. This tutorial covers self-hosted deployments only. While this tutorial uses Docker Compose, the same security features and configurations apply to any self-hosted deployment method."
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
  "anchor": "step-1-start-an-unsecured-instance",
  "chunk_num": 0,
  "text": "Step 1: Start an Unsecured Instance Start Qdrant using the standard Docker Compose setup. Create a `docker-compose.yml` file: Start the instance: Confirm that no credentials are required when connecting to the REST API port with `curl`: Expected response: No api-key header was required. Anyone who can reach this port can read, write, or delete all data. ---"
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
  "anchor": "step-2-enable-tls",
  "chunk_num": 0,
  "text": "Step 2: Enable TLS Unencrypted connections allow anyone on the network to read your API key and data in transit. Enable TLS to encrypt all traffic. First, add a local certificate authority to your system trust store, so `curl` and your browser will accept the certificate without extra flags. Next, generate a locally trusted certificate with mkcert: If you're using the Python or TypeScript clients, set the following environment variables to allow the clients to find the certificate: If you're using the Java client, add the certificate to the Java trust store: Next, update `docker-compose.yml` to enable TLS and mount the certificate files: Restart Qdrant to apply the changes: Now, unencrypted HTTP requests are rejected: However, HTTPS requests succeed: Refer to Security > TLS to learn more about TLS configuration. ---"
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
  "anchor": "step-3-enable-an-admin-api-key",
  "chunk_num": 0,
  "text": "Step 3: Enable an Admin API Key Without enabling authentication, anyone with network access to a Qdrant instance can read, write, or delete all its data. Set an admin API key to require credentials on every request. Set the `QDRANT__SERVICE__API_KEY` environment variable to the API key in `docker-compose.yml`: Restart Qdrant to apply the changes: Verify that unauthenticated requests are now rejected: The same behavior applies to the clients. Ingesting a point without an API key is blocked: With the admin API key, the request succeeds: Refer to Security > Authentication to learn more about admin API keys, including API key rotation. ---"
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
  "anchor": "step-4-enable-a-read-only-api-key",
  "chunk_num": 0,
  "text": "Step 4: Enable a Read-Only API Key Issue a separate read-only API key for services that only need to read data. With this key, a client application can search and read but cannot upsert, delete, or modify data. Set the `QDRANT__SERVICE__READ_ONLY_API_KEY` environment variable to the read-only key in `docker-compose.yml`: Restart Qdrant: Verify that a delete attempt with the read-only key is rejected: Or with a client: Reads still succeed with the read-only key: Both keys can be used simultaneously. See Security > Read-Only API Key. ---"
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
  "anchor": "step-5-set-up-granular-access-api-keys-jwt",
  "chunk_num": 0,
  "text": "Step 5: Set Up Granular Access API Keys (JWT) The admin and read-only keys apply globally. For finer control, use granular access API Keys (JSON Web Tokens, JWT). For example, you can use JWT to provide read-write access to one collection and read-only access to another. Enable JWT RBAC in `docker-compose.yml`: Restart: Create a second collection `other_collection` using the admin API key: Generate a JWT in the Web UI: 1. Open `https://localhost:6333/dashboard#/jwt`. If you get a warning about the connection not being private, this is because the certificate is self-signed. If so, restart the browser, and it should recognize the certificate as trusted. 1. Select **Collection Access**. 1. For `my_collection`, select **Read** and **Write**. 1. For `other_collection`, select **Read** only. 1. Copy the generated JWT Token. Generating a JWT token with the desired access levels using the Web UI."
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
  "anchor": "step-5-set-up-granular-access-api-keys-jwt",
  "chunk_num": 1,
  "text": "Step 5: Set Up Granular Access API Keys (JWT) > JWT tokens can also be generated programmatically. See Security > Granular Access API Keys for a list of libraries that can be used to generate JWT tokens. Using the JWT token, writing to `my_collection` (`rw` scope) should succeed: With a client too: However, writing to `other_collection` (`r` scope) is blocked: With a client too: See Security > Granular Access Control with JWT for the full list of available JWT claims and the complete access-level table. ---"
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
  "anchor": "whats-next",
  "chunk_num": 0,
  "text": "What's Next Your instance now has TLS encryption, API key authentication, a read-only key for query consumers, and collection-scoped JWT tokens. For production deployments, also consider: - Network Bind — restrict which network interfaces Qdrant listens on. - API Key Rotation — rotate admin API keys in a distributed deployment without downtime. - Production Checklist — a full checklist of security and reliability settings for production."
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/migration/",
  "anchor": "conclusion",
  "chunk_num": 0,
  "text": "Conclusion The **Qdrant Migration Tool** makes data transfer across vector database instances effortless. Whether you're moving between cloud regions, upgrading from self-hosted to Qdrant Cloud, or switching from other databases such as Pinecone, this tool saves you hours of manual effort. Try it today. For detailed per-provider migration guides (Pinecone, Weaviate, Milvus, Elasticsearch, pgvector), see the Migrate to Qdrant section. After migrating, use the Migration Verification Guide to confirm data integrity and search quality."
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/migration/",
  "anchor": "example-migrate-from-pinecone-to-qdrant",
  "chunk_num": 0,
  "text": "Example: Migrate from Pinecone to Qdrant Let’s now walk through an example of migrating from Pinecone to Qdrant. Assume your Pinecone index looks like this: The information you need from Pinecone is: * Your Pinecone API key * The index name * The index host URL With that information, you can migrate your vector database from Pinecone to Qdrant with the following command: When the migration is complete, you will see the new collection on Qdrant with all the vectors."
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/migration/",
  "anchor": "how-to-use-the-qdrant-migration-tool",
  "chunk_num": 0,
  "text": "How to Use the Qdrant Migration Tool You can run the tool via Docker. Installation: Here is an example of how to perform a Qdrant to Qdrant migration: Note: The migration CLI uses the Qdrant gRPC API, so you must always configure the gRPC port for Qdrant URLs with the Migration CLI (default: 6334)."
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/migration/",
  "anchor": "migrate-your-embeddings-to-qdrant",
  "chunk_num": 0,
  "text": "Migrate Your Embeddings to Qdrant | Time: Varies | Level: Intermediate | | --- | ----------- | Migrating data between vector databases, especially across regions, platforms, or deployment types, can be a hassle. That’s where the Qdrant Migration Tool comes in. It supports a wide range of migration needs, including transferring data between Qdrant instances and migrating from other vector database providers to Qdrant. You can run the migration tool on any machine where you have connectivity to both the source and the target Qdrant databases. Direct connectivity between both databases is not required. For optimal performance, you should run the tool on a machine with a fast network connection and minimum latency to both databases. In this tutorial, we will learn how to use the migration tool and walk through a practical example of migrating from another vector database to Qdrant."
 },
 {
  "url": "https://qdrant.tech/documentation/tutorials-operations/migration/",
  "anchor": "why-use-this-instead-of-qdrants-native-snapshotting",
  "chunk_num": 0,
  "text": "Why use this instead of Qdrant’s Native Snapshotting? Qdrant supports snapshot-based backups, which are low-level disk operations built for same-cluster recovery or local backups. These snapshots: * Require snapshot consistency across nodes. * Can be hard to port across machines or cloud zones. On the other hand, the Qdrant Migration Tool: * Streams data in live batches. * Can resume interrupted migrations. * Works even when data is being inserted. * Supports collection reconfiguration (e.g., changing replication settings and quantization). * Supports migrating from other vector DBs (Pinecone, Chroma, Weaviate, etc.)"
 },
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "conclusion",
 "chunk_num": 0,
 "text": "Conclusion Time-based sharding is a powerful technique for managing large, time-series datasets in Qdrant. By routing data to different shards based on timestamps, you can efficiently store and query recent data while easily pruning older data without impacting performance. This approach is ideal for use cases like social media analysis, where data relevance decreases over time."
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "create-collection",
 "chunk_num": 0,
 "text": "Create Collection Create a collection with user-defined sharding by setting the sharding method to custom. Custom shards can be accessed by their shard key. In this tutorial, the shard keys are the dates in `YYYY-MM-DD` format, extracted from the timestamp of each data point. This collection will have a single shard for each shard key (a separate shard for each day of data). For very large datasets, you can improve write throughput by configuring a `shard_number` for the collection. `shard_number` defaults to 1. Set it to a higher value to create multiple shards per shard key to distribute the write load across multiple peers in the cluster. However, avoid creating too many shards, as each shard consumes resources and adds overhead, which can lead to performance degradation. Test what the optimal number of shards is for your dataset and cluster configuration."
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "create-collection",
 "chunk_num": 1,
 "text": "Create Collection Two things to note about collections that use user-defined sharding versus regular collections using auto sharding: - For regular collections using auto sharding, `shard_number` determines the total number of shards for the collection. However, with user-defined sharding, `shard_number` determines the number of shards **per shard key**: the total number of shards for a collection equals the number of shard keys (days) multiplied by the `shard_number`. - Collection-level configuration changes that you can apply to a regular collection (for example, HNSW parameters) can also be applied to a collection with user-defined sharding. These changes are applied retroactively to existing shards and to new shards created in the future."
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "ingest-historical-data",
 "chunk_num": 0,
 "text": "Ingest Historical Data Time series data often arrives in streams, with new data points continuously being added. Each data point may include a timestamp indicating when it was created. You can use this timestamp to determine which shard a data point belongs to. If your data does not have timestamps, you can use the current time. This tutorial uses a sample dataset of social media posts with timestamps. Let's assume today is April 7th, 2026. You'll start by ingesting some historical data from this week (April 1-7). Here are a few sample rows from the dataset: | datetime | text |---|---| | 2026-04-06T09:04:28 | April sunshine through the office window makes everything better. | 2026-04-06T09:04:32 | Morning stretch, good coffee, clear intentions. Monday: sorted. | 2026-04-06T09:05:52 | Grateful for a productive first day of the week. Upload the dataset and store each day of data in its own shard: Let's break down the code:"
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "ingest-historical-data",
 "chunk_num": 1,
 "text": "Ingest Historical Data - First, a list of existing shard keys in the collection is retrieved. There should be none because you just created the collection, but in production, this ensures you take into account any existing data. - Next, a CSV file is streamed from a URL using a helper function to parse the CSV. Details"
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "ingest-historical-data",
 "chunk_num": 2,
 "text": "Ingest Historical Data - The CSV file is streamed row by row, buffering points in batches of 100 for efficient uploading. The optimal batch size depends on your data and cluster, so you may want to experiment with different sizes for best performance. - The date (`YYYY-MM-DD`) is extracted from each row's datetime field. This date is used as the shard key to route the data to the correct shard. - A new shard is created for each new date encountered if it doesn't already exist. - The buffer is flushed to the previous date's shard whenever the date changes mid-stream, ensuring posts don't get written to the wrong shard. - Data is written, where each point gets: - A random UUID as the point ID - The post `text` and `datetime` as the payload - A dense vector embedding generated from the post text using `sentence-transformers/all-MiniLM-L6-v2` - Each full batch of 100 points is uploaded to Qdrant, targeting the correct date-based shard via the shard key selector parameter."
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "ingest-historical-data",
 "chunk_num": 3,
 "text": "Ingest Historical Data - Any remaining points are uploaded in a partial final batch after the loop ends."
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "ingest-new-data",
 "chunk_num": 0,
 "text": "Ingest New Data When ingesting new data, set the `shard_key_selector` to today's date so the data goes to the correct shard:"
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "install-and-initialize-the-qdrant-client",
 "chunk_num": 0,
 "text": "Install and Initialize the Qdrant Client First, install the Qdrant client: Next, initialize the client: This tutorial assumes you are using Qdrant Cloud Inference to generate vector embeddings. If you manage your own embedding infrastructure, you can apply the same principles, but you will need to adapt the code examples to use your embedding service."
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "pruning-shards",
 "chunk_num": 0,
 "text": "Pruning Shards Every night at midnight, create a new shard for the new data that will be ingested that day. If you only query the last 7 days of data, you can also delete the oldest shard. You can automate this with a cron job."
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "query-multiple-days-of-data",
 "chunk_num": 0,
 "text": "Query Multiple Days of Data To query multiple shards, set the shard key selector to a list of shard keys. For example, to query the last 2 days of data (April 6-7):"
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "query-the-full-dataset",
 "chunk_num": 0,
 "text": "Query the Full Dataset To query the entire dataset (all shards), omit the shard key selector parameter:"
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "query-todays-data",
 "chunk_num": 0,
 "text": "Query Today's Data Now you can run a semantic query on the posts. Setting the shard key selector to `2026-04-07` (assuming today is April 7th, 2026) limits the query to the shard that stores today's data:"
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "time-based-sharding-in-qdrant",
 "chunk_num": 0,
 "text": "Time-Based Sharding in Qdrant When working with massive, fast-moving datasets, like social media or image/video streams, efficient storage and retrieval are critical. Often, only the most recent data is relevant, while older data can be archived or deleted. For instance, in sentiment analysis of social media posts, you might only need the last 7 days of data to capture current trends, with most queries focusing on the last 24 hours. Storing everything in Qdrant collection with default sharding can lead to expensive re-indexing across the entire dataset when deleting old points, impacting performance. A better solution is **time-based sharding**, where points are routed to a specific shard (or shards) based on timestamp. For use cases with a natural time-to-live (TTL) segmentation, sharding by a timestamp-based key enables efficient querying of recent data and allows users to seamlessly drop the old."
},
{
 "url": "https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/",
 "anchor": "time-based-sharding-in-qdrant",
 "chunk_num": 1,
 "text": "Time-Based Sharding in Qdrant For example, with daily shards, today's data is stored in today's shard, yesterday's data in yesterday's shard, and so on. Queries can target specific shards (today's shard, for example) or multiple shards to cover a date range. Time-based sharding routes data to different shards based on timestamp. Typically, all writes go to the newest shard, while queries can target one or more shards. Older shards can be pruned in the background without affecting performance. Depending on your data volume and retention needs, you could shard by hour, week, month, or any other time interval that suits your use case. This tutorial guides you through implementing time-based sharding and covers: * Creating a Qdrant collection with user-defined sharding * Batch-ingesting historical data into the correct shards based on timestamps * Assigning new data to the most recent shard * Querying one or more shards * Pruning older shards"
}
]

print(len(CHUNKS), "chunks; one of them:")
print(CHUNKS[0])

30 chunks; one of them:
{'url': 'https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/', 'anchor': 'prerequisites', 'chunk_num': 0, 'text': 'Prerequisites - Docker and Docker Compose installed - `curl` available in your terminal - mkcert for generating a local self-signed certificate (installation instructions) - TLS requires Qdrant 1.2 or later, API key authentication requires Qdrant 1.2 or later, and granular access API keys (JWT) require Qdrant 1.9 or later. This tutorial uses the latest Qdrant image, which includes all these features. ---'}


For each chunk we assume some text normalization pipeline is in place, as:

- Noise in the text degrades the embedding
- Noise costs re-embedding when it's not needed (e.g. someone added a trailing space)

For example:

In [16]:
import re
import unicodedata

def normalize(text):
    # unify visually-identical unicode forms (a non-breaking space vs a regular space)
    text = unicodedata.normalize("NFKC", text)

    # remove invisible characters: zero-width spaces and joiners, byte-order mark, soft hyphen
    text = text.translate(dict.fromkeys(map(ord, "\u200b\u200c\u200d\ufeff\u00ad")))

    # collapse any whitespace run into a single space
    return re.sub(r"\s+", " ", text).strip()

## Configuring Documentation Collection

Let's configure a collection for chunks.

We'll use `sentence-transformers/all-minilm-l6-v2`: it's free on [Qdrant Cloud Inference](https://qdrant.tech/documentation/inference/cloud-inference/). Its output dimension is 384, its context window is 256 tokens, which is exactly why long sections got chunked above: over-window input is truncated silently, and a truncated embedding is one more way to quietly degrade search.

### Collection Metadata

**Note:** There are other types of drift harmful for production vector search, for example, the embedding model version changes, or the data preparation pipeline.
Vectors produced by different embedding models, or by the same model over differently prepared text, almost certainly should not mix in one collection: retrieval will degrade and it will be hard to detect why.

Let's consider a simple guardrail: save which model and which pipeline version produced the data points, in [**collection metadata**](https://qdrant.tech/documentation/manage-data/collections/#collection-metadata), and verify against it. If one of the two changed, we need to trigger full collection re-embedding, not fix temporal drift point-wise.

In [17]:
MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # free on Qdrant Cloud Inference
PIPELINE = "docs-prep-pipeline-v1"                 # versions pipeline that delivers the vectorized text
COLLECTION = "docs-sync-tutorial"

In [ ]:
client.create_collection(
    COLLECTION,
    vectors_config=models.VectorParams(
        size=384,  # all-minilm-l6-v2 output dimension
        distance=models.Distance.COSINE,
    ),
    metadata={"embedding_model": MODEL, "pipeline_version": PIPELINE},
)

True

In [19]:
def check_gate():
    # compare this pipeline's constants against what the collection records about itself
    meta = client.get_collection(COLLECTION).config.metadata or {}

    if meta.get("embedding_model") != MODEL or meta.get("pipeline_version") != PIPELINE:
        raise RuntimeError(f"collection was built by {meta}: full re-embed into a fresh collection required")


check_gate()

## One Point in Documentation Collection

### Documentation Change Characteristics

What usually happens to documentation?
Something completely new appears. Information on pages gets fixed. Pages get restructured and sections are moved as-is: same content, new address. And of course something gets deleted.

The process can be reflected by monitoring two independent characteristics of a document chunk:

- **Content**: the text we search against and generate the embedding from.
- **Position**: where the chunk lives, in our case its URL, anchor, and number.

Hence every record should get two derived values:

- **Content fingerprint** for content. For example, SHA-256 of the text. It changes if a single character changes, and never otherwise [*]. Comparing fingerprints answers "Is it the same content?" without comparing texts.
- **Deterministic ID** for position. For example, `url + "#" + anchor + "::" + chunk_num` turned into a UUID, as UUID is one of the two point ID formats Qdrant accepts. Comparing IDs answers "Is this content still at the same position?".

[*] **Note:** One could try to save on re-embeddings even more, checking semantic difference between the old and new text against some tuned threshold. We use exact content identity deliberately. A minor edit may cause a re-embedding, but this synchronization pipeline does not need to decide whether two different texts are “close enough.”

In [20]:
import hashlib
import uuid
from datetime import datetime, timezone

def content_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()

def point_id(url, anchor, num):
    # NAMESPACE_URL is a fixed constant uuid5 requires; it marks the input as a URL-like name
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{url}#{anchor}::{num}"))

def prepare_chunks_for_sync(chunks):
    """Derive both values (and the section address) for every raw chunk."""
    out = []
    for c in chunks:
        text = normalize(c["text"])
        out.append({
            **c,
            "text": text,
            "section_url": f"{c['url']}#{c['anchor']}" if c["anchor"] else c["url"],
            "content_hash": content_hash(text),
            "point_id": point_id(c["url"], c["anchor"], c["chunk_num"]),
        })
    return out

In [21]:
example = prepare_chunks_for_sync(CHUNKS[:1])[0]

print("point ID (from the address):", example["point_id"])
print("text (to vectorize):        ", example["text"][:60], "...")
print("content_hash:               ", example["content_hash"])

point ID (from the address): 2ff5204a-0353-5991-ba55-acd1995063e8
text (to vectorize):         Prerequisites - Docker and Docker Compose installed - `curl` ...
content_hash:                27d55e75b962f1d5cd11b8136623889f157303917ba697d9a0e876f805280be7


### Vector and Metadata

One point in the documentation collection then carries the following:

**ID:**
UUID(`url + "#" + anchor + "::" + chunk_num`)

**Payload:**
- `url`, `anchor`, `chunk_num`: recompute the point ID, link to the source
  - also `url`: filter or group all chunks of one page
- `section_url`: filter or group all chunks of one section
- `text`: the exact embedded string; what gets re-embedded on the next change
- `content_hash`: the raw data change detector
- `last_updated`: when content of this chunk last changed (or was created)

**Vector:**
- `text` embedded with `all-minilm-l6-v2`

In [22]:
def payload(chunk, last_updated=None):
    return {
        "url": chunk["url"],
        "anchor": chunk["anchor"],
        "chunk_num": chunk["chunk_num"],
        "section_url": chunk["section_url"],
        "text": chunk["text"],
        "content_hash": chunk["content_hash"],
        "last_updated": last_updated or datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

For all the payload fields used for filtering or grouping we need to create a [**payload index**](https://qdrant.tech/documentation/manage-data/indexing/). New collections on Qdrant Cloud run in strict mode: filtering on a non-indexed field returns an error, not a slow scan. Index everything you will filter on:

In [23]:
for field in ("content_hash", "url", "section_url"):
    client.create_payload_index(COLLECTION, field, models.PayloadSchemaType.KEYWORD)

## Populate the Collection

Let's quickly check how search over our documentation collection consisting of the CHUNKS above would look:

In [24]:
client.upsert(COLLECTION, points=[
    models.PointStruct(
        id=c["point_id"],
        vector=models.Document(text=c["text"], model=MODEL),  # Cloud Inference embeds text server-side
        payload=payload(c),
    )
    for c in prepare_chunks_for_sync(CHUNKS)
], wait=True)

print(client.count(COLLECTION).count, "points in the collection")

30 points in the collection


In [25]:
import textwrap

QUERY = "Where exactly to set `QDRANT__SERVICE__API_KEY` variable to enable authentication for a self-hosted Qdrant?"
query_embedding = models.Document(text=QUERY, model=MODEL)

def show(results):
    for p in results.points:
        print(f"{p.score:.3f}  {p.payload['section_url']}")
        print(textwrap.fill(p.payload["text"], width=100,
                            initial_indent="       ", subsequent_indent="       "))
        print()


show(
    client.query_points(
        COLLECTION,
        query=query_embedding,
        limit=3,
        with_payload=["section_url", "text"],
    )
)

0.675  https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/#secure-a-self-hosted-qdrant-instance
       Secure a Self-Hosted Qdrant Instance | Time: 45 min | Level: Intermediate | | --- |
       ----------- | Qdrant offers a comprehensive set of security and access control features that
       enable you to protect your data and control access at multiple levels. By default, these
       features are enabled on Qdrant Cloud deployments. However, self-hosted Qdrant deployments
       default to no authentication and no encryption: every interface on the host is reachable
       without a key or password. For self-hosted instances, it is crucial to secure your instance
       before connecting it to any network. This tutorial walks through securing a self-hosted
       Qdrant instance step by step. You will: - **Enable TLS** to encrypt traffic between clients
       and your Qdrant instance. - **Set up an admin API key** to require authentication for all
       requests.

## Syncing with Documentation Changes

Your sync trigger could be a CI job on merge if your docs live in git or rely on a **cron** (run once a night).

The input of every sync with a documentation collection is the **current full chunk list of the docs**. For a simple data prep pipeline like the one above it's cheap to gather this full list once a day, saving the headache of deriving raw changes.

Each incoming chunk compares to the current documentation collection in one of the following ways, based on the point ID (the chunk's address in documentation) and `content_hash` (the chunk's exact content, its fingerprint):

- **unchanged**: same ID, same fingerprint → the point stays as is
- **content changed**: same ID, new fingerprint → re-embed in place: update vector, `content_hash`, `last_updated`
- **address changed**: no such ID in the collection, but an identical content hash exists → the embedding can be reused (the text may have just moved); we write it to a point with the new ID, payload and vector stay the same
- **new**: new chunk in the incoming list, both ID- and content-wise → embed and insert a new point
- **gone**: the collection has a chunk whose ID is absent from the incoming sync list → delete

**Note:** Optional footgun-guard: take a [snapshot](https://qdrant.tech/documentation/concepts/snapshots/) before sync, delete it later when everything looks fine.

### Input of a Sync Pipeline

Let's consider some possible changes:

- adding to the ["Secure a Self-Hosted Qdrant Instance" tutorial](https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/) a new small section **Step 6: Rotate API keys**, with the **Step 3** section now pointing to it → *one new chunk + one embedding changed in place*
- the [**migration** page](https://qdrant.tech/documentation/tutorials-operations/migration) moved to a new URL, from `migration` to `migration-guide` → *new IDs, same vectors/texts*
- the ["Time-based sharding" tutorial](https://qdrant.tech/documentation/tutorials-operations/time-based-sharding/) was removed

In [26]:
untouched_secure_qdrant = [
    c for c in CHUNKS
    if c["url"] == "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/"
    and c["anchor"] != "step-3-enable-an-admin-api-key"
]

# now points to the new section
step_3 = {
 "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
 "anchor": "step-3-enable-an-admin-api-key",
 "chunk_num": 0,
 "text": "Step 3: Enable an Admin API Key Without enabling authentication, anyone with network access to a Qdrant instance can read, write, or delete all its data. Set an admin API key to require credentials on every request. Set the `QDRANT__SERVICE__API_KEY` environment variable to the API key in `docker-compose.yml`: Restart Qdrant to apply the changes: Verify that unauthenticated requests are now rejected: The same behavior applies to the clients. Ingesting a point without an API key is blocked: With the admin API key, the request succeeds: Refer to Security > Authentication to learn more about admin API keys, including API key rotation. --- See also: rotating API keys."
}

# the new section
step_6 = {
 "url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
 "anchor": "step-6-rotate-api-keys",
 "chunk_num": 0,
 "text": "Step 6: Rotate API keys Rotate the admin API key on a schedule and immediately after any suspected exposure. Update every client before revoking the old key."
}

# the migration page moved: same texts, new addresses
moved = [
    {**c, "url": "https://qdrant.tech/documentation/tutorials-operations/migration-guide/"}
    for c in CHUNKS
    if c["url"] == "https://qdrant.tech/documentation/tutorials-operations/migration/"
]

# the time-based-sharding tutorial is absent from LATEST_CHUNKS - that is how a deletion arrives

LATEST_CHUNKS = prepare_chunks_for_sync(untouched_secure_qdrant + [step_3, step_6] + moved)

print(len(LATEST_CHUNKS), "chunks in the latest crawl")

16 chunks in the latest crawl


We now check every incoming chunk against the collection: does its ID (address) exist, and does its `content_hash` (exact text) match? 

[`retrieve`](https://qdrant.tech/documentation/concepts/points/) fetches points by ID. At corpus scale you would batch the IDs.

In [27]:
def split_by_state(latest_chunks):
    """Compare the incoming chunk list to the collection: who is unchanged, changed, or unknown."""
    incoming = {c["point_id"]: c for c in latest_chunks}

    stored = {}
    points = client.retrieve(
        COLLECTION,
        ids=list(incoming),
        with_payload=["content_hash"],
        with_vectors=False,
    )
    for p in points:
        stored[str(p.id)] = p.payload["content_hash"]

    unchanged = [c for pid, c in incoming.items() if stored.get(pid) == c["content_hash"]]
    content_changed = [c for pid, c in incoming.items() if pid in stored and stored[pid] != c["content_hash"]]
    unknown_ids = [c for pid, c in incoming.items() if pid not in stored]

    return incoming, unchanged, content_changed, unknown_ids


incoming_ids, unchanged, content_changed, unknown_ids = split_by_state(LATEST_CHUNKS)

print(f"unchanged: {len(unchanged)}   changed text: {len(content_changed)}   unknown IDs: {len(unknown_ids)}")

unchanged: 9   changed text: 1   unknown IDs: 6


### Case 1: Unchanged, Do Nothing

These chunks carry the same fingerprint as before.

### Case 2: Content Changed, Re-Embed

The chunk about Step 3 exists under a known ID (it didn't change its position on the docs website) but carries new information.  
Use `upsert`: writing a point under an existing ID replaces it.

In [28]:
def re_embed_changed(content_changed):
    if not content_changed:
        return
    client.upsert(COLLECTION, points=[
        models.PointStruct(
            id=c["point_id"],
            vector=models.Document(text=c["text"], model=MODEL),
            payload=payload(c),
        )
        for c in content_changed
    ], wait=True)


re_embed_changed(content_changed)
print("re-embedded:", len(content_changed))

re-embedded: 1


### Cases 3 and 4: ID Is Not Present in the Collection

Six IDs are unknown to the collection, but an unknown ID does not necessarily mean new content. When a page moves as-is, every chunk on it gets a new address, a new ID, while the text stays exactly the same. Embedding it again would produce the same vector, so why pay for it.

A filtered [`scroll`](https://qdrant.tech/documentation/concepts/points/) on `content_hash` answers the question "does this exact text already exist under some other ID?". On a hit, we copy the stored vector into the new point and keep the source's `last_updated`: the content did not change, only its address did.

**Note** *This version performs one hash lookup per unknown chunk so the decision is easy to inspect. In production, batch hash lookups and point upserts.*

On a miss, the content is genuinely new; we embed and insert a new point.

In [29]:
def reuse_or_add(unknown_ids):
    """Reuse an existing embedding when the same text is already stored; embed only what is new."""
    reused, added = 0, 0

    for c in unknown_ids:
        same_text = models.Filter(must=[
            models.FieldCondition(
                key="content_hash",
                match=models.MatchValue(value=c["content_hash"]),
            )
        ])
        hits, _ = client.scroll(
            COLLECTION,
            scroll_filter=same_text,
            limit=1,
            with_payload=["last_updated"],
            with_vectors=True,
        )

        if hits:  # same text, new address: copy the vector, keep its last_updated
            point = models.PointStruct(
                id=c["point_id"],
                vector=hits[0].vector,
                payload=payload(c, hits[0].payload["last_updated"]),
            )
            reused += 1
        else:     # genuinely new content: embed and insert
            point = models.PointStruct(
                id=c["point_id"],
                vector=models.Document(text=c["text"], model=MODEL),
                payload=payload(c),
            )
            added += 1

        client.upsert(COLLECTION, points=[point], wait=True)

    return reused, added


reused, added = reuse_or_add(unknown_ids)
print(f"embeddings reused: {reused}   embedded as new: {added}")

embeddings reused: 5   embedded as new: 1


The five chunks of the moved migration page kept their embeddings for free; only Step 6, the new section, was embedded.

What's important to notice: the old points, the migration page under its old URL, are still in the collection. They need to be removed, and that is the last case.

### Case 5: Gone, Delete (Last in Order)

Whatever LATEST_CHUNKS does not contain no longer exists at the source. The deletion is one filtered call, "every point whose ID is *not* in the incoming list".

**Note:** *Deletion runs **last**, after all writes: hence if someone queries documentation at night while this pipeline runs, results for a second might be weird, as mid-run search here sees old and new content side by side:)*

**Note:** *It's a good practice to put some guardrails on the number of deletions before running it: if it is suspiciously large, you might want to skip deletion and investigate instead. Mind the edge case: an empty incoming list would match every point in the collection, so refuse to sync empty input.*

**Note:** Frequent re-embeddings and deletions don't degrade the index over time: background [optimizers](https://qdrant.tech/documentation/ops-optimization/optimizer/) rebuild and merge index segments as changes accumulate.


In [ ]:
def delete_gone(incoming_ids):
    """Remove every point the current crawl no longer contains. Returns how many."""
    if not incoming_ids:
        raise ValueError("Refusing to delete from an empty source snapshot.")

    stale = models.Filter(must_not=[models.HasIdCondition(has_id=list(incoming_ids))])

    to_delete = client.count(COLLECTION, count_filter=stale).count

    # potential check against a threshold to avoid accidental mass deletion could be added here
    print("deleting", to_delete, "points") 
    client.delete(COLLECTION, points_selector=models.FilterSelector(filter=stale), wait=True)
    return to_delete


deleted = delete_gone(incoming_ids)

deleting 20 points


## Verify the Sync

The same query: the edited Step 3 surfaces with its new text.

In [31]:
show(
    client.query_points(
        COLLECTION,
        query=models.Document(text=QUERY, model=MODEL),
        limit=3,
        with_payload=["section_url", "text"],
    )
)

0.678  https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/#step-3-enable-an-admin-api-key
       Step 3: Enable an Admin API Key Without enabling authentication, anyone with network access
       to a Qdrant instance can read, write, or delete all its data. Set an admin API key to require
       credentials on every request. Set the `QDRANT__SERVICE__API_KEY` environment variable to the
       API key in `docker-compose.yml`: Restart Qdrant to apply the changes: Verify that
       unauthenticated requests are now rejected: The same behavior applies to the clients.
       Ingesting a point without an API key is blocked: With the admin API key, the request
       succeeds: Refer to Security > Authentication to learn more about admin API keys, including
       API key rotation. --- See also: rotating API keys.

0.675  https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/#secure-a-self-hosted-qdrant-instance
       Secure a Self-Hosted Qdrant Instance | Ti

### All Together in One `sync()`

In [32]:
def sync(latest_chunks):
    check_gate()  # refuse to mix embedding models or pipeline versions

    chunks = prepare_chunks_for_sync(latest_chunks)
    incoming_ids, unchanged, content_changed, unknown_ids = split_by_state(chunks)

    re_embed_changed(content_changed)
    reused, added = reuse_or_add(unknown_ids)
    deleted = delete_gone(incoming_ids)

    return {
        "unchanged": len(unchanged),
        "re-embedded": len(content_changed),
        "reused_embedding": reused,
        "added": added,
        "deleted": deleted,
    }

A re-run of the same sync input should change nothing: every counter at zero. It's a cheap assertion that everything is synced.

In [33]:
rerun = sync(LATEST_CHUNKS)
print(rerun)

assert rerun["re-embedded"] == rerun["added"] == rerun["reused_embedding"] == rerun["deleted"] == 0

deleting 0 points
{'unchanged': 16, 're-embedded': 0, 'reused_embedding': 0, 'added': 0, 'deleted': 0}


## Conclusion

A reliable sync separates location from content:

- deterministic IDs identify a chunk’s location,
- content hashes detect text changes,
- existing vectors can be reused when identical text moves, and
- deletion reconciles the collection with an authoritative source snapshot.

Ways to make it better:

- Pipelines that risk concurrent iterative updates: see [conditional updates](https://qdrant.tech/documentation/manage-data/points/#conditional-updates) and [update modes](https://qdrant.tech/documentation/manage-data/points/#update-mode), per-write preconditions. This pipeline runs one sync at a time and does not need them.
- The `last_updated` field this sync maintains can power recency-aware ranking via [decay functions in FormulaQuery](https://qdrant.tech/documentation/search/search-relevance/).

Related guides: 
- Switching or upgrading the embedding model by [Embedding Model Migration](https://qdrant.tech/documentation/tutorials-operations/embedding-model-migration/)
- Wholesale infrastructure swaps by [Blue-Green Deployment](https://qdrant.tech/documentation/tutorials-operations/blue-green-deployment/)
- Sync driven by database change events by [Data Synchronization](https://qdrant.tech/documentation/data-synchronization/).